# Fine-tune SAM Base (ViT-B) cho hạt lúa

Pipeline này thay thế pipeline SAM 3 và nạp checkpoint `MAIN_SOURCES/MODELS/sam_b.pt`. Dữ liệu dùng cùng Roboflow project/version với YOLO nhưng export COCO instance segmentation.

SAM-B là promptable segmenter, không phải detector. Train/validation/test dùng ground-truth bounding box làm prompt, nên metric đo chất lượng mask khi đã có box. Khi triển khai phải cấp box từ YOLO hoặc detector khác. Ultralytics không hỗ trợ high-level training cho `sam_b.pt`; notebook fine-tune mask decoder bằng vòng PyTorch riêng.

In [ ]:
# Bước 1: Mount Drive và cấu hình
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
BASE_PATH = Path('/content/drive/MyDrive/NGHIÊN CỨU KHOA HỌC/GROUP_MEMBERS/NGUYEN MINH TRI/MAIN_SOURCES')
MODEL_PATH = BASE_PATH / 'MODELS' / 'sam_b.pt'
ROBOFLOW_WORKSPACE = 'minh-tr-tklp0'
ROBOFLOW_PROJECT = 'full-grains-segmentaion'
ROBOFLOW_VERSION = 1
IMAGE_SIZE, MASK_SIZE = 1024, 256
EPOCHS, LEARNING_RATE, WEIGHT_DECAY = 30, 1e-5, 1e-4
PROMPT_CHUNK_SIZE, BOX_JITTER_FRACTION = 32, 0.03
PATIENCE, SEED, NUM_WORKERS = 8, 123, 2
RUN_NAME = f'{ROBOFLOW_PROJECT}-v{ROBOFLOW_VERSION}_sam_b_decoder_ft'
LOCAL_RUN_DIR = Path('/content/sam_base_runs') / RUN_NAME
DRIVE_RUN_DIR = BASE_PATH / 'RESULTS' / RUN_NAME
LOCAL_RUN_DIR.mkdir(parents=True, exist_ok=True)
if not MODEL_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy {MODEL_PATH}. Hãy đặt sam_b.pt đúng trong MAIN_SOURCES/MODELS.')
print('Checkpoint:', MODEL_PATH)

In [ ]:
# Bước 2: Dependencies và import module training của project
%pip install -q 'ultralytics>=8.4.120,<8.5.0' roboflow pycocotools opencv-python-headless matplotlib pandas

import sys
sys.path.insert(0, str(BASE_PATH / 'CODE'))
import json, math, random, shutil
from collections import Counter
from datetime import datetime, timezone
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import ultralytics
from google.colab import userdata
from torch.utils.data import DataLoader
from ultralytics.models.sam.build import build_sam
from modules.sam_base_trainer import (
    CocoSamPromptDataset, freeze_for_decoder_finetune, run_epoch,
    save_ultralytics_sam_checkpoint, single_item_collate,
)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
assert torch.cuda.is_available(), 'Hãy chọn GPU runtime trong Colab.'
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
DEVICE = torch.device('cuda:0')
GPU_NAME = torch.cuda.get_device_name(0)
AMP_DTYPE = torch.bfloat16 if torch.cuda.get_device_capability(0)[0] >= 8 else torch.float16
print('GPU:', GPU_NAME, '| torch:', torch.__version__, '| ultralytics:', ultralytics.__version__)

In [ ]:
# Bước 3: Tải COCO instance segmentation từ đúng Roboflow version
from roboflow import Roboflow
ROBOFLOW_API_KEY = userdata.get('ROBOFLOW_API_KEY')
if not ROBOFLOW_API_KEY:
    raise RuntimeError('Thiếu Colab Secret ROBOFLOW_API_KEY.')
REQUESTED_DATASET_DIR = Path('/content/datasets/full-grains-segmentaion-v1-coco')
if (REQUESTED_DATASET_DIR / 'train/_annotations.coco.json').is_file():
    DATASET_DIR = REQUESTED_DATASET_DIR.resolve()
else:
    rf = Roboflow(api_key=ROBOFLOW_API_KEY)
    version = rf.workspace(ROBOFLOW_WORKSPACE).project(ROBOFLOW_PROJECT).version(ROBOFLOW_VERSION)
    dataset = version.download('coco-segmentation', location=str(REQUESTED_DATASET_DIR))
    DATASET_DIR = Path(dataset.location).resolve()
print('Dataset:', DATASET_DIR)

In [ ]:
# Bước 4: Kiểm tra split/annotation trước khi train
def inspect_split(split):
    split_dir = DATASET_DIR / split
    ann_path = split_dir / '_annotations.coco.json'
    if not ann_path.is_file():
        raise FileNotFoundError(ann_path)
    data = json.loads(ann_path.read_text(encoding='utf-8'))
    if not data.get('images') or not data.get('annotations'):
        raise ValueError(f'{split}: images/annotations rỗng')
    missing = [x['file_name'] for x in data['images'] if not (split_dir / x['file_name']).is_file()]
    invalid = [x.get('id') for x in data['annotations'] if not x.get('segmentation') or len(x.get('bbox', [])) != 4]
    if missing or invalid:
        raise ValueError(f'{split}: missing_images={len(missing)}, invalid_annotations={len(invalid)}')
    counts = Counter(x['image_id'] for x in data['annotations'])
    per_image = [counts.get(x['id'], 0) for x in data['images']]
    return {'images': len(data['images']), 'instances': len(data['annotations']), 'mean_instances': float(np.mean(per_image)), 'max_instances': int(max(per_image))}
DATASET_SUMMARY = {split: inspect_split(split) for split in ('train', 'valid', 'test')}
print(json.dumps(DATASET_SUMMARY, indent=2))

In [ ]:
# Bước 5: Dataset/DataLoader; batch ảnh = 1, hạt trong ảnh được chia prompt chunks
common = dict(image_size=IMAGE_SIZE, mask_size=MASK_SIZE, box_jitter_fraction=BOX_JITTER_FRACTION)
train_dataset = CocoSamPromptDataset(DATASET_DIR, 'train', training=True, **common)
valid_dataset = CocoSamPromptDataset(DATASET_DIR, 'valid', training=False, **common)
test_dataset = CocoSamPromptDataset(DATASET_DIR, 'test', training=False, **common)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, num_workers=NUM_WORKERS, collate_fn=single_item_collate, generator=generator, pin_memory=True)
valid_loader = DataLoader(valid_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=single_item_collate, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False, num_workers=NUM_WORKERS, collate_fn=single_item_collate, pin_memory=True)
sample = next(iter(valid_loader))
assert sample['image'].shape == (3, IMAGE_SIZE, IMAGE_SIZE)
assert sample['masks'].shape[-2:] == (MASK_SIZE, MASK_SIZE)
assert sample['boxes'].ndim == 2 and sample['boxes'].shape[1] == 4
print('Dataset smoke test PASS; validation instances in sample:', len(sample['boxes']))

In [ ]:
# Bước 6: Nạp đúng sam_b.pt, đóng băng encoder và train mask decoder
sam_model = build_sam(str(MODEL_PATH)).to(DEVICE)
trainable_parameters = freeze_for_decoder_finetune(sam_model)
optimizer = torch.optim.AdamW(trainable_parameters, lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2, min_lr=1e-7)
scaler = torch.amp.GradScaler('cuda', enabled=True)
total_params = sum(p.numel() for p in sam_model.parameters())
trainable_params = sum(p.numel() for p in trainable_parameters)
print(f'Trainable: {trainable_params:,}/{total_params:,} ({100*trainable_params/total_params:.2f}%)')
assert 0 < trainable_params < total_params

In [ ]:
# Bước 7: Train, early stopping và lưu best/last checkpoint
BEST_WEIGHT = LOCAL_RUN_DIR / 'rice_sam_b.pt'
LAST_WEIGHT = LOCAL_RUN_DIR / 'last_rice_sam_b.pt'
HISTORY_CSV = LOCAL_RUN_DIR / 'history.csv'
metadata = {'workspace': ROBOFLOW_WORKSPACE, 'project': ROBOFLOW_PROJECT, 'version': ROBOFLOW_VERSION, 'seed': SEED}
history, best_iou, stale_epochs = [], -1.0, 0
for epoch in range(1, EPOCHS + 1):
    train_metrics = run_epoch(sam_model, train_loader, DEVICE, AMP_DTYPE, PROMPT_CHUNK_SIZE, True, optimizer, scaler, trainable_parameters)
    valid_metrics = run_epoch(sam_model, valid_loader, DEVICE, AMP_DTYPE, PROMPT_CHUNK_SIZE, False)
    scheduler.step(valid_metrics['iou'])
    row = {'epoch': epoch, 'learning_rate': optimizer.param_groups[0]['lr'], **{f'train_{k}': v for k, v in train_metrics.items()}, **{f'valid_{k}': v for k, v in valid_metrics.items()}}
    history.append(row); pd.DataFrame(history).to_csv(HISTORY_CSV, index=False)
    save_ultralytics_sam_checkpoint(LAST_WEIGHT, sam_model, optimizer, epoch, valid_metrics, metadata)
    if valid_metrics['iou'] > best_iou:
        best_iou, stale_epochs = valid_metrics['iou'], 0
        save_ultralytics_sam_checkpoint(BEST_WEIGHT, sam_model, optimizer, epoch, valid_metrics, metadata)
    else:
        stale_epochs += 1
    print(f'Epoch {epoch:03d} | train loss={train_metrics["loss"]:.4f} IoU={train_metrics["iou"]:.4f} | valid loss={valid_metrics["loss"]:.4f} IoU={valid_metrics["iou"]:.4f} Dice={valid_metrics["dice"]:.4f}')
    if stale_epochs >= PATIENCE:
        print('Early stopping'); break
assert BEST_WEIGHT.is_file() and BEST_WEIGHT.stat().st_size > 0
print('Best validation IoU:', best_iou, '|', BEST_WEIGHT)

In [ ]:
# Bước 8: Đánh giá test đúng một lần và lưu evidence
checkpoint = torch.load(BEST_WEIGHT, map_location='cpu', weights_only=False)
sam_model.load_state_dict(checkpoint['model'], strict=True); sam_model.to(DEVICE)
test_metrics = run_epoch(sam_model, test_loader, DEVICE, AMP_DTYPE, PROMPT_CHUNK_SIZE, False)
result = {
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'source_model': str(MODEL_PATH), 'model_type': 'SAM Base ViT-B',
    'training_scope': 'mask_decoder_only',
    'evaluation_protocol': 'ground_truth_box_prompt_per_instance',
    'best_validation_iou': best_iou, 'test_metrics': test_metrics,
    'dataset_summary': DATASET_SUMMARY, 'gpu': GPU_NAME,
    'pytorch_version': torch.__version__, 'ultralytics_version': ultralytics.__version__,
}
(LOCAL_RUN_DIR / 'result.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(test_metrics, indent=2))

In [ ]:
# Bước 9: Learning curves và sao lưu lên Drive
frame = pd.read_csv(HISTORY_CSV)
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for axis, metric, title in zip(axes, ('loss', 'iou', 'dice'), ('Loss', 'Prompted mask IoU', 'Prompted mask Dice')):
    axis.plot(frame['epoch'], frame[f'train_{metric}'], label='train')
    axis.plot(frame['epoch'], frame[f'valid_{metric}'], label='valid')
    axis.set_title(title); axis.set_xlabel('Epoch'); axis.grid(alpha=.25); axis.legend()
plt.tight_layout(); plt.savefig(LOCAL_RUN_DIR / 'training_curves.png', dpi=160, bbox_inches='tight'); plt.show()
DRIVE_RUN_DIR.mkdir(parents=True, exist_ok=True)
shutil.copytree(LOCAL_RUN_DIR, DRIVE_RUN_DIR, dirs_exist_ok=True)
required = [DRIVE_RUN_DIR / 'rice_sam_b.pt', DRIVE_RUN_DIR / 'history.csv', DRIVE_RUN_DIR / 'result.json', DRIVE_RUN_DIR / 'training_curves.png']
missing = [str(path) for path in required if not path.is_file()]
if missing: raise FileNotFoundError(f'Sao lưu thiếu: {missing}')
print('Export:', DRIVE_RUN_DIR)
print('Fine-tuned model:', DRIVE_RUN_DIR / 'rice_sam_b.pt')

## Sử dụng sau khi train

`rice_sam_b.pt` có thể nạp bằng `from ultralytics import SAM; model = SAM(path)` vì tên file kết thúc bằng `sam_b.pt` và checkpoint có trường `model` đúng loader Ultralytics. Inference vẫn cần box/point prompt.

Không so trực tiếp test IoU ở đây với YOLO mAP: SAM-B đang nhận ground-truth boxes. So sánh toàn pipeline phải chạy `YOLO boxes → SAM-B masks` trên cùng test split rồi tính mask AP, count error, thời gian và VRAM.